# DL_3D vs DL_2D ROC comparison

In [1]:

# ============================================================
# DL_3D vs DL_2D ROC comparison
# Outputs: individual ROC curves for train/internal/external and
# one 1 x 3 combined figure. Saves both PDF and SVG.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.metrics import roc_auc_score, roc_curve

# ============================================================
# 1. Paths and style
# ============================================================

model_root = Path('/host/d/projects/Habitats/models/Prognosis')
results_out_dir = Path('/host/d/projects/Habitats/results/DL_3D2D_COMPARISON')
results_out_dir.mkdir(parents=True, exist_ok=True)

# Use the same manuscript font style as AUC_reports.ipynb.
times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

cohort_specs = {
    'train': {
        'display': 'Train',
        'file': 'cv_final_selection_predictions.xlsx',
    },
    'internal_test': {
        'display': 'Internal test',
        'file': 'internal_test_final_selection_predictions.xlsx',
    },
    'external_test': {
        'display': 'External test',
        'file': 'external_test_final_selection_predictions.xlsx',
    },
}

model_specs = {
    'DL_3D': {
        'folder': model_root / 'dl_3d_ml_all' / 'final_selections',
        'color': '#B279A2',
    },
    'DL_2D': {
        'folder': model_root / 'dl_2d_ml_all' / 'final_selections',
        'color': '#4C78A8',
    },
}

label_col = 'Prognosis_label'
prob_col = 'prob_final_selection'

print('Output directory:', results_out_dir)


# ============================================================
# 2. Load final-selection predictions
# ============================================================

def load_prediction_table(model_name, cohort_key):
    path = model_specs[model_name]['folder'] / cohort_specs[cohort_key]['file']
    if not path.is_file():
        raise FileNotFoundError(path)

    df = pd.read_excel(path)
    missing_cols = [col for col in [label_col, prob_col] if col not in df.columns]
    if missing_cols:
        raise KeyError(f'{path} missing columns: {missing_cols}')

    y_true = df[label_col].astype(int).to_numpy()
    y_prob = df[prob_col].astype(float).to_numpy()
    auc = roc_auc_score(y_true, y_prob)

    return {
        'path': path,
        'df': df,
        'y_true': y_true,
        'y_prob': y_prob,
        'auc': auc,
    }

all_predictions = {
    cohort_key: {
        model_name: load_prediction_table(model_name, cohort_key)
        for model_name in model_specs
    }
    for cohort_key in cohort_specs
}

for cohort_key, cohort_predictions in all_predictions.items():
    print('\n' + cohort_specs[cohort_key]['display'])
    for model_name, pred in cohort_predictions.items():
        print(f"  {model_name}: AUC={pred['auc']:.3f}, n={len(pred['y_true'])}, file={pred['path']}")


# ============================================================
# 3. Plot helpers
# ============================================================

def save_pdf_and_svg(fig, pdf_path, **kwargs):
    pdf_path = Path(pdf_path)
    svg_path = pdf_path.with_suffix('.svg')
    fig.savefig(pdf_path, **kwargs)
    fig.savefig(svg_path, **kwargs)
    print('Saved:', pdf_path)
    print('Saved:', svg_path)


def style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.35)


def plot_roc_for_cohort(cohort_key, ax=None, show_title=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
    else:
        fig = ax.figure

    for model_name, spec in model_specs.items():
        pred = all_predictions[cohort_key][model_name]
        fpr, tpr, _ = roc_curve(pred['y_true'], pred['y_prob'])
        ax.plot(
            fpr,
            tpr,
            color=spec['color'],
            linewidth=1.9,
            label=f"{model_name}: AUC {pred['auc']:.3f}",
        )

    ax.plot([0, 1], [0, 1], color='#888888', linewidth=1.0, linestyle='--')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('1 - Specificity', fontsize=14)
    ax.set_ylabel('Sensitivity', fontsize=14)
    if show_title:
        ax.set_title(f"ROC curve: {cohort_specs[cohort_key]['display']}", fontsize=17, fontweight='bold')
    ax.legend(loc='lower right', fontsize=13, frameon=False, borderaxespad=0.2)
    style_axes(ax)
    return fig, ax

print('Plot helpers ready.')


# ============================================================
# 4. Save individual and combined figures
# ============================================================

individual_paths = {
    'train': results_out_dir / 'ROC_DL_3D_vs_2D_train.pdf',
    'internal_test': results_out_dir / 'ROC_DL_3D_vs_2D_internal_test.pdf',
    'external_test': results_out_dir / 'ROC_DL_3D_vs_2D_external_test.pdf',
}

for cohort_key, out_path in individual_paths.items():
    fig, ax = plot_roc_for_cohort(cohort_key, show_title=True)
    fig.tight_layout()
    save_pdf_and_svg(fig, out_path, bbox_inches='tight')
    plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8))
ordered_cohorts = ['train', 'internal_test', 'external_test']
for ax, cohort_key in zip(axes, ordered_cohorts):
    plot_roc_for_cohort(cohort_key, ax=ax, show_title=False)
    ax.set_title(cohort_specs[cohort_key]['display'], fontsize=18, fontweight='bold')

fig.tight_layout(w_pad=1.4)
combined_path = results_out_dir / 'ROC_DL_3D_vs_2D_combined.pdf'
save_pdf_and_svg(fig, combined_path, bbox_inches='tight')
plt.close(fig)

print('\nDone.')


Output directory: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON



Train
  DL_3D: AUC=0.844, n=188, file=/host/d/projects/Habitats/models/Prognosis/dl_3d_ml_all/final_selections/cv_final_selection_predictions.xlsx
  DL_2D: AUC=0.804, n=188, file=/host/d/projects/Habitats/models/Prognosis/dl_2d_ml_all/final_selections/cv_final_selection_predictions.xlsx

Internal test
  DL_3D: AUC=0.815, n=96, file=/host/d/projects/Habitats/models/Prognosis/dl_3d_ml_all/final_selections/internal_test_final_selection_predictions.xlsx
  DL_2D: AUC=0.795, n=96, file=/host/d/projects/Habitats/models/Prognosis/dl_2d_ml_all/final_selections/internal_test_final_selection_predictions.xlsx

External test
  DL_3D: AUC=0.792, n=64, file=/host/d/projects/Habitats/models/Prognosis/dl_3d_ml_all/final_selections/external_test_final_selection_predictions.xlsx
  DL_2D: AUC=0.772, n=64, file=/host/d/projects/Habitats/models/Prognosis/dl_2d_ml_all/final_selections/external_test_final_selection_predictions.xlsx
Plot helpers ready.


Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_train.pdf
Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_train.svg


Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_internal_test.pdf
Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_internal_test.svg


Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_external_test.pdf
Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_external_test.svg


Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_combined.pdf
Saved: /host/d/projects/Habitats/results/DL_3D2D_COMPARISON/ROC_DL_3D_vs_2D_combined.svg

Done.
